<a href="https://colab.research.google.com/github/HARSITHRAM/Intelligent-Textile-Inspection-System/blob/main/model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ==========================================
# 1. INSTALL DEPENDENCIES
# ==========================================
!pip install -q anomalib
!pip install -q kaggle opencv-python-headless matplotlib torch torchvision

import os
import shutil
import glob
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import torch

# Verify GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cpu


In [3]:
# ==========================================
# 2. DATASET DOWNLOAD FROM KAGGLE
# ==========================================
from google.colab import files

# Prompt to upload kaggle.json
if not os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")):
    print("Please upload your 'kaggle.json' file downloaded from Kaggle Account Settings.")
    uploaded = files.upload()
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    for fn in uploaded.keys():
        shutil.move(fn, os.path.expanduser("~/.kaggle/kaggle.json"))
    !chmod 600 ~/.kaggle/kaggle.json

# Download AITEX dataset
!kaggle datasets download -d nexuswho/aitex-fabric-image-database
!unzip -q aitex-fabric-image-database.zip -d aitex_raw
print("Dataset downloaded and extracted successfully!")

Dataset URL: https://www.kaggle.com/datasets/nexuswho/aitex-fabric-image-database
License(s): Attribution-NonCommercial-NoDerivatives 4.0 International (CC BY-NC-ND 4.0)
aitex-fabric-image-database.zip: Skipping, found more recently modified local copy (use --force to force download)
replace aitex_raw/Defect_images/0001_002_00.png? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace aitex_raw/Defect_images/0002_002_00.png? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace aitex_raw/Defect_images/0003_002_00.png? [y]es, [n]o, [A]ll, [N]one, [r]ename: 
error:  invalid response [{ENTER}]
replace aitex_raw/Defect_images/0003_002_00.png? [y]es, [n]o, [A]ll, [N]one, [r]ename: 
error:  invalid response [{ENTER}]
replace aitex_raw/Defect_images/0003_002_00.png? [y]es, [n]o, [A]ll, [N]one, [r]ename: 
error:  invalid response [{ENTER}]
replace aitex_raw/Defect_images/0003_002_00.png? [y]es, [n]o, [A]ll, [N]one, [r]ename: 
error:  invalid response [{ENTER}]
replace aitex_raw/Defect_images/0003_002_00.p

In [4]:
# ==========================================
# 3. CONVERT AITEX TO MVTEC STRUCTURE
# ==========================================
base_dir = "Fabric_MVTec"
train_good_dir = os.path.join(base_dir, "fabric", "train", "good")
test_good_dir = os.path.join(base_dir, "fabric", "test", "good")
test_defect_dir = os.path.join(base_dir, "fabric", "test", "defect")
gt_defect_dir = os.path.join(base_dir, "fabric", "ground_truth", "defect")

# Create directory structure
for d in [train_good_dir, test_good_dir, test_defect_dir, gt_defect_dir]:
    os.makedirs(d, exist_ok=True)

# Parse AITEX structure (AITEX provides defect-free and defective images)
# 'nod缺陷样本' typically contains normal, while '缺陷样本' contains defects
all_images = glob.glob("aitex_raw/**/*.png", recursive=True) + glob.glob("aitex_raw/**/*.jpg", recursive=True)

normal_imgs = [f for f in all_images if "nodefect" in f.lower() or "0000" in os.path.basename(f)]
defect_imgs = [f for f in all_images if f not in normal_imgs and "mask" not in f.lower()]
mask_imgs = [f for f in all_images if "mask" in f.lower()]

# Split normal images into Train and Test Good (80/20 split)
split_idx = int(len(normal_imgs) * 0.8)
train_normal = normal_imgs[:split_idx]
test_normal = normal_imgs[split_idx:]

for img in train_normal:
    shutil.copy(img, os.path.join(train_good_dir, os.path.basename(img)))
for img in test_normal:
    shutil.copy(img, os.path.join(test_good_dir, os.path.basename(img)))

# Map defects and their corresponding ground-truth masks
for img in defect_imgs:
    img_name = os.path.basename(img)
    shutil.copy(img, os.path.join(test_defect_dir, img_name))

    # Try to locate corresponding mask based on naming conventions
    base_name = os.path.splitext(img_name)[0]
    expected_mask = [m for m in mask_imgs if base_name in os.path.basename(m)]
    if expected_mask:
        shutil.copy(expected_mask[0], os.path.join(gt_defect_dir, img_name))
    else:
        # Generate an empty mask or placeholder if mask is missing
        placeholder = np.zeros((4096, 256), dtype=np.uint8) # Default AITEX scale reference
        cv2.imwrite(os.path.join(gt_defect_dir, img_name), placeholder)

print(f"MVTec formatting complete.\nTrain normal: {len(train_normal)} | Test normal: {len(test_normal)} | Test defect: {len(defect_imgs)}")

MVTec formatting complete.
Train normal: 112 | Test normal: 29 | Test defect: 106


In [ ]:
# ==========================================
# 4. TRAINING EFFICIENTAD (FIXED RECURSION)
# ==========================================
import os
from anomalib.data import MVTecAD
from anomalib.models import EfficientAd
from anomalib.engine import Engine
from torchvision.transforms import v2

# Import standard text-based UI components to prevent Colab recursion crash
from lightning.pytorch.callbacks import TQDMProgressBar, ModelSummary

# 1. Define the resize parameter
transform = v2.Compose([
    v2.Resize(size=(256, 256), antialias=True)
])

# 2. Initialize Dataset DataModule
datamodule = MVTecAD(
    root=base_dir,
    category="fabric",
    train_batch_size=1,
    eval_batch_size=8,
    num_workers=2,
    augmentations=transform
)
datamodule.setup()

# 3. Instantiate EfficientAD Model
model = EfficientAd()

# 4. Configure Engine/Trainer Configuration
# FIX: Pass standard callbacks to bypass the 'rich' UI library conflict in Colab
engine = Engine(
    max_epochs=50,
    accelerator="auto",
    devices=1,
    default_root_dir="./results",
    callbacks=[TQDMProgressBar(), ModelSummary()]  # <-- Prevents the infinite loop
)

# 5. Start training loop
print("Beginning training phase on normal fabric samples...")
engine.fit(model=model, datamodule=datamodule)
print("Training finalized!")

INFO:lightning_fabric.utilities.rank_zero:Trainer already configured with model summary callbacks: [<class 'lightning.pytorch.callbacks.model_summary.ModelSummary'>]. Skipping setting a default `ModelSummary` callback.
INFO:lightning_fabric.utilities.rank_zero:GPU available: False, used: False
INFO:lightning_fabric.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:lightning_fabric.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 
  | Name           | Type             | Params | Mode  | FLOPs
--------------------------------------------------------------------
0 | pre_processor  | PreProcessor     | 0      | train | 0    
1 | post_processor | PostProcessor    | 0      | train | 0    
2 | evaluator      | Evaluator        | 0      | train | 0    
3 | model          | E

Beginning training phase on normal fabric samples...


/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/loops/fit_loop.py:538: Found 7 module(s) in eval mode at the start of training. This may lead to unexpected behavior during training. If this is intentional, you can ignore this warning.


Training: |          | 0/? [00:00<?, ?it/s]


efficientad_pretrained_weights.zip: 0.00B [00:00, ?B/s]
efficientad_pretrained_weights.zip:   0%|          | 8.19k/40.0M [00:00<30:25, 21.9kB/s]
efficientad_pretrained_weights.zip:  15%|█▌        | 6.04M/40.0M [00:00<00:02, 16.7MB/s]
efficientad_pretrained_weights.zip:  26%|██▋       | 10.5M/40.0M [00:00<00:01, 23.8MB/s]
efficientad_pretrained_weights.zip:  30%|███       | 12.1M/40.0M [00:00<00:01, 21.4MB/s]
efficientad_pretrained_weights.zip:  37%|███▋      | 14.9M/40.0M [00:00<00:01, 23.6MB/s]
efficientad_pretrained_weights.zip:  53%|█████▎    | 21.0M/40.0M [00:00<00:00, 27.8MB/s]
efficientad_pretrained_weights.zip:  65%|██████▌   | 26.1M/40.0M [00:01<00:00, 33.7MB/s]
efficientad_pretrained_weights.zip:  77%|███████▋  | 30.6M/40.0M [00:01<00:00, 36.5MB/s]
efficientad_pretrained_weights.zip:  79%|███████▊  | 31.5M/40.0M [00:01<00:00, 25.4MB/s]
efficientad_pretrained_weights.zip: 40.0MB [00:01, 26.6MB/s]                            
/usr/local/lib/python3.12/dist-packages/torchvision/t

...

Calculate Validation Dataset Quantiles: 100%|██████████| 17/17 [00:59<00:00,  3.52s/it]


Validation: |          | 0/? [00:00<?, ?it/s]

Calculate Validation Dataset Quantiles: 100%|██████████| 17/17 [00:59<00:00,  3.53s/it]


Validation: |          | 0/? [00:00<?, ?it/s]

Calculate Validation Dataset Quantiles: 100%|██████████| 17/17 [00:59<00:00,  3.50s/it]


In [ ]:
# ==========================================
# 5. INFERENCE, POST-PROCESSING & HOLE COUNTING
# ==========================================
import glob
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
from PIL import Image
from torchvision.transforms import v2

output_results_dir = "./hole_detection_results"
os.makedirs(output_results_dir, exist_ok=True)

# 1. Match the exact transformation pipeline used during training
transform = v2.Compose([
    v2.Resize(size=(256, 256), antialias=True),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True)
])

# 2. Prepare the model for inference
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# 3. Locate defective test images
test_defect_dir = os.path.join(base_dir, "fabric", "test", "defect")
defect_test_images = glob.glob(os.path.join(test_defect_dir, "*"))

for idx, img_path in enumerate(defect_test_images[:5]): # Processing first 5 for visualization
    filename = os.path.basename(img_path)

    # Load original image
    orig_img = Image.open(img_path).convert("RGB")
    input_tensor = transform(orig_img).unsqueeze(0).to(device)

    # Run Inference
    with torch.no_grad():
        output = model(input_tensor)

    # Safely extract the anomaly map (Handles different return types in Anomalib v2)
    if isinstance(output, dict):
        anomaly_map = output.get("anomaly_map", next(iter(output.values()))).squeeze().cpu().numpy()
    elif hasattr(output, "anomaly_map"):
        anomaly_map = output.anomaly_map.squeeze().cpu().numpy()
    else:
        # Fallback if the model returns the raw tensor directly
        anomaly_map = output.squeeze().cpu().numpy()

    # --- OpenCV Blob Segmentation & Feature Metrics ---
    # Normalize anomaly map strictly between 0 and 255
    anomaly_map_norm = (anomaly_map - anomaly_map.min()) / (anomaly_map.max() - anomaly_map.min() + 1e-8)
    gray_anomaly = np.uint8(255 * anomaly_map_norm)

    # Resize back to standard 256x256 if needed and apply color map
    gray_anomaly_resized = cv2.resize(gray_anomaly, (256, 256))
    heatmap = cv2.applyColorMap(gray_anomaly_resized, cv2.COLORMAP_JET)

    # Thresholding configuration (Separates severe anomalies like holes from background)
    _, thresh = cv2.threshold(gray_anomaly_resized, 128, 255, cv2.THRESH_BINARY)

    # Morphological Opening (Cleans up tiny noise pixels)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    clean_mask = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel)

    # Find Contours
    contours, _ = cv2.findContours(clean_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Initialize working canvas
    display_img = cv2.resize(np.array(orig_img), (256, 256))
    bbox_img = display_img.copy()

    hole_count = 0
    print(f"\n--- Analysis Results for File: {filename} ---")

    for c in contours:
        area = cv2.contourArea(c)
        if area < 15:  # Ignore tiny artifacts
            continue

        hole_count += 1

        # Draw Bounding Box
        x, y, w, h = cv2.boundingRect(c)
        cv2.rectangle(bbox_img, (x, y), (x + w, y + h), (255, 0, 0), 2)

        # Calculate & Draw Centroid
        M = cv2.moments(c)
        if M["m00"] != 0:
            cX = int(M["m10"] / M["m00"])
            cY = int(M["m01"] / M["m00"])
        else:
            cX, cY = x + w//2, y + h//2

        cv2.circle(bbox_img, (cX, cY), 4, (0, 255, 0), -1)

        print(f"Hole {hole_count}")
        print(f"Area = {int(area)} pixels")
        print(f"Centroid = ({cX}, {cY})\n")

    print(f"Total holes = {hole_count}\n")

    # --- Visualization ---
    fig, axes = plt.subplots(1, 5, figsize=(20, 5))
    axes[0].imshow(display_img)
    axes[0].set_title("Original Image")

    axes[1].imshow(cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB))
    axes[1].set_title("Anomaly Heatmap")

    axes[2].imshow(clean_mask, cmap='gray')
    axes[2].set_title("Binary Mask")

    axes[3].imshow(bbox_img)
    axes[3].set_title("Bounding Boxes")

    axes[4].imshow(bbox_img)
    axes[4].set_title(f"Final Detected Holes: {hole_count}")

    for ax in axes:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

    # Save Outputs
    cv2.imwrite(os.path.join(output_results_dir, f"{filename}_orig.png"), cv2.cvtColor(display_img, cv2.COLOR_RGB2BGR))
    cv2.imwrite(os.path.join(output_results_dir, f"{filename}_heatmap.png"), heatmap)
    cv2.imwrite(os.path.join(output_results_dir, f"{filename}_mask.png"), clean_mask)
    cv2.imwrite(os.path.join(output_results_dir, f"{filename}_final.png"), cv2.cvtColor(bbox_img, cv2.COLOR_RGB2BGR))